# Train Whisper — Severe Impairment
Single-phase fine-tune of `openai/whisper-large-v3` on **severe** impairment data only.
- Higher augmentation probability (`AUG_PROB=0.7`) compensates for smaller dataset
- `DeferredEarlyStopping` prevents cosine-LR warmup dips from stopping training early
- `label_smoothing_factor=0.1` improves generalisation on small datasets

## Cell 1: Install packages

In [1]:
import subprocess, sys

packages = [
    "huggingface_hub==0.23.4", "datasets==2.19.2", "peft==0.11.1",
    "accelerate>=0.30.0", "transformers>=4.39.0", "evaluate", "jiwer",
    "soundfile", "librosa", "audiomentations", "pyroomacoustics",
    "numpy", "matplotlib", "scipy",
    "better-profanity",
]
print("Installing packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], capture_output=True)
all_ok = True
for pkg in packages:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    ok = r.returncode == 0
    print(f"  {'OK' if ok else 'FAIL'} {pkg}")
    if not ok:
        all_ok = False
        print(f"       {r.stderr[-300:].strip()}")
print("\nAll packages installed." if all_ok else "\nSome installs failed.")

Installing packages...
  OK huggingface_hub==0.23.4
  OK datasets==2.19.2
  OK peft==0.11.1
  OK accelerate>=0.30.0
  OK transformers>=4.39.0
  OK evaluate
  OK jiwer
  OK soundfile
  OK librosa
  OK audiomentations
  OK pyroomacoustics
  OK numpy
  OK matplotlib
  OK scipy
  OK better-profanity

All packages installed.


## Cell 2: Imports

In [2]:
import os, sys, json, re, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from collections import defaultdict
from threading import Thread

from datasets import load_dataset, Audio, concatenate_datasets
from huggingface_hub import login, hf_hub_download
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer,
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    EarlyStoppingCallback, TextIteratorStreamer,
    pipeline as hf_pipeline,
)
from peft import LoraConfig, get_peft_model
import evaluate
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# ── DeferredEarlyStopping ────────────────────────────────────────────────────
class DeferredEarlyStopping(EarlyStoppingCallback):
    """Skips early-stop checks for the first `min_epochs` epochs so the
    cosine LR warmup phase cannot trigger a premature stop."""
    def __init__(self, min_epochs: int = 3, **kwargs):
        super().__init__(**kwargs)
        self._min_epochs = min_epochs

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if state.epoch is not None and state.epoch < self._min_epochs:
            return
        super().on_evaluate(args, state, control, metrics, **kwargs)

print("Imports OK")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Imports OK
  PyTorch : 2.8.0+cu129
  CUDA    : True
  GPU     : NVIDIA A100-SXM4-80GB
  VRAM    : 85.1 GB


## Cell 3: Settings

In [3]:
# ── Auth ──────────────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Storage ────────────────────────────────────────────────────────────────
LOCAL_STORAGE_DIR = '/jupyter_kernel'
OUTPUT_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models', 'whisper_severe')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Dataset ────────────────────────────────────────────────────────────────
DATASET_NAME     = "cdli/ugandan_english_nonstandard_speech_v1.0"
AUDIO_COL        = "audio"
TRANSCRIPT_COL   = "transcription"
SPEAKER_COL      = "speaker_id"
PROMPT_TYPE_COL  = "prompt_type"
SAMPLE_RATE      = 16_000
LANGUAGE         = 'en'
TASK             = 'transcribe'

# ── Severity target ────────────────────────────────────────────────────────
SEVERITY_LABEL  = "severe"
SEVERITY_FILTER = ["severe"]

# ── Base model ─────────────────────────────────────────────────────────────
WHISPER_MODEL_TYPE = "openai/whisper-large-v3"

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R        = 32
LORA_ALPHA    = 64
LORA_DROPOUT  = 0.1
LORA_TARGETS  = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]

# ── Augmentation (raised vs original 0.3 to compensate for small dataset) ─
AUG_PROB      = 0.7

# ── Trainer ────────────────────────────────────────────────────────────────
MAX_EPOCHS      = 25
LEARNING_RATE   = 1e-5
BATCH_SIZE      = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUM      = 4
EARLY_STOP_PAT  = 7
LR_WARMUP_STEPS = 50
MAX_GEN_LEN     = 225
NUM_CHECKPOINTS = 5
LOGGING_STEPS   = 20

USE_BF16 = torch.cuda.is_available()
USE_FP16 = False

# ── HF Hub ─────────────────────────────────────────────────────────────────
REPO_NAME = "jojo007unfi/whisper-severe"

# ── Modal ──────────────────────────────────────────────────────────────────
MODAL_VOLUME_EXPORT_DIR = "/asr-models/whisper-severe"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Severity target : {SEVERITY_LABEL}")
print(f"Device          : {device}")
print(f"LoRA            : rank={LORA_R}  alpha={LORA_ALPHA}")
print(f"Augmentation    : AUG_PROB={AUG_PROB}")
print(f"Training        : LR={LEARNING_RATE}  epochs={MAX_EPOCHS}  patience={EARLY_STOP_PAT}")
print(f"Output dir      : {OUTPUT_DIR}")
print(f"HF repo         : {REPO_NAME}")

Severity target : severe
Device          : cuda
LoRA            : rank=32  alpha=64
Augmentation    : AUG_PROB=0.7
Training        : LR=1e-05  epochs=25  patience=7
Output dir      : /jupyter_kernel/trained_models/whisper_severe
HF repo         : jojo007unfi/whisper-severe


## Cell 4: HuggingFace login

In [4]:
if not HF_TOKEN:
    HF_TOKEN = input("Paste your HuggingFace token: ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in.


## Cell 5: Augmentation config

In [5]:
from audiomentations import (
    Compose, AddGaussianNoise, AddGaussianSNR,
    PitchShift, TimeStretch, Gain,
    LowPassFilter, RoomSimulator,
)

AUGMENT_CONFIG = {
    "min_snr_db": 10.0, "max_snr_db": 40.0, "p_snr":     0.5,
    "min_amp":   0.001, "max_amp":   0.015,  "p_noise":   0.4,
    "min_semi":   -2.0, "max_semi":    2.0,  "p_pitch":   0.5,
    "min_rate":   0.85, "max_rate":   1.15,  "p_stretch": 0.5,
    "min_sx": 3.6, "max_sx": 5.6,
    "min_sy": 3.6, "max_sy": 3.9,
    "min_sz": 2.4, "max_sz": 3.0,
    "min_srcx": 0.1, "max_srcx": 3.5,
    "min_srcy": 0.1, "max_srcy": 2.7,
    "min_srcz": 1.0, "max_srcz": 2.1,
    "min_mic":  0.15, "max_mic":  0.35, "p_room":  0.4,
    "min_lp": 3000.0, "max_lp": 7000.0, "p_lp":   0.3,
    "min_gain": -6.0, "max_gain":  6.0, "p_gain":  0.5,
}
print(f"Augmentation config ready (AUG_PROB={AUG_PROB})")

Augmentation config ready (AUG_PROB=0.7)


## Cell 6: Load dataset & filter to severity

In [6]:
PARQUET_FILES = {
    "train":      [f"data/train-{str(i).zfill(5)}-of-00008.parquet" for i in range(8)],
    "validation": ["data/validation-00000-of-00001.parquet"],
    "test":       ["data/test-00000-of-00002.parquet", "data/test-00001-of-00002.parquet"],
    "extra":      ["data/extra-00000-of-00001.parquet"],
}

print("Downloading parquet shards...")
local_files = {}
for split, files in PARQUET_FILES.items():
    local_files[split] = []
    for fname in files:
        print(f"  {split}/{fname.split('/')[-1]} ...", end=" ", flush=True)
        path = hf_hub_download(
            repo_id=DATASET_NAME, filename=fname,
            repo_type="dataset", token=HF_TOKEN,
        )
        local_files[split].append(path)
        print("OK")

raw = load_dataset("parquet", data_files=local_files)

HAVE_METADATA = False
speaker_meta  = None
try:
    meta_path    = hf_hub_download(repo_id=DATASET_NAME, filename="speaker_metadata.csv",
                                   repo_type="dataset", token=HF_TOKEN)
    speaker_meta = pd.read_csv(meta_path)
    HAVE_METADATA = True
    print(f"Speaker metadata loaded: {list(speaker_meta.columns)}")
except Exception as e:
    print(f"Metadata not loaded: {e}")

# ── Merge all non-test data into one pool, keep test separate ────────────────
# We do our OWN speaker-disjoint split so validation speakers never appear
# in training (no leakage) and train count always exceeds val count.
full_pool = concatenate_datasets([raw["train"], raw["extra"], raw["validation"]])
test_ds   = concatenate_datasets([raw["test"]])

# Filter to <=30s
AUDIO_LEN_COL = "audio_length"
for split_name, ds_ref in [("pool", full_pool), ("test", test_ds)]:
    n_before = len(ds_ref)
    if AUDIO_LEN_COL in ds_ref.column_names:
        if split_name == "pool":
            full_pool = full_pool.filter(lambda x: x[AUDIO_LEN_COL] <= 30)
            print(f"  pool: {n_before} -> {len(full_pool)} (<=30s filter)")
        else:
            test_ds = test_ds.filter(lambda x: x[AUDIO_LEN_COL] <= 30)
            print(f"  test: {n_before} -> {len(test_ds)} (<=30s filter)")

# ── Build severity map ────────────────────────────────────────────────────────
spk_to_severity = {}
if HAVE_METADATA and speaker_meta is not None:
    spk_to_severity = dict(
        zip(speaker_meta["speaker_id"].astype(str),
            speaker_meta["severity_speech_impairment"].astype(str))
    )
    print(f"Severity map built for {len(spk_to_severity)} speakers.")

def filter_severity(dataset, keywords):
    if not spk_to_severity:
        print("WARNING: No severity map — using full dataset.")
        return dataset
    def keep(ex):
        sev = spk_to_severity.get(str(ex[SPEAKER_COL]), "unknown").lower()
        return any(k.lower() in sev for k in keywords)
    filtered = dataset.filter(keep, num_proc=1)
    print(f"  Filtered {len(dataset):,} -> {len(filtered):,} samples for {keywords}")
    return filtered

# Filter the full pool to this notebook's severity tier
severity_pool = filter_severity(full_pool, SEVERITY_FILTER)

# ── Speaker-disjoint train / val split (80 / 20 by speaker) ─────────────────
import random as _random
_random.seed(42)

all_speakers = sorted(set(str(ex[SPEAKER_COL]) for ex in severity_pool))
_random.shuffle(all_speakers)

n_val_spk      = max(1, int(len(all_speakers) * 0.20))
val_speakers   = set(all_speakers[:n_val_spk])
train_speakers = set(all_speakers[n_val_spk:])

train_ds = severity_pool.filter(
    lambda ex: str(ex[SPEAKER_COL]) in train_speakers, num_proc=1)
val_ds   = severity_pool.filter(
    lambda ex: str(ex[SPEAKER_COL]) in val_speakers,   num_proc=1)

# Guard: if val somehow ends up larger, move val speakers to train one-by-one
_sorted_val = sorted(val_speakers)
_idx = 0
while len(train_ds) <= len(val_ds) and _idx < len(_sorted_val):
    move = _sorted_val[_idx]; _idx += 1
    val_speakers.discard(move)
    train_speakers.add(move)
    train_ds = severity_pool.filter(
        lambda ex: str(ex[SPEAKER_COL]) in train_speakers, num_proc=1)
    val_ds   = severity_pool.filter(
        lambda ex: str(ex[SPEAKER_COL]) in val_speakers,   num_proc=1)

# ── Sanity checks (will raise immediately if violated) ───────────────────────
_overlap = train_speakers & val_speakers
assert not _overlap,          f"Speaker overlap detected: {_overlap}"
assert len(train_ds) > 0,     "Training set is empty after severity filter!"
assert len(val_ds)   > 0,     "Validation set is empty after severity filter!"
assert len(train_ds) > len(val_ds), (
    f"train ({len(train_ds)}) must be > val ({len(val_ds)})")

print(f"\nSpeaker-disjoint split for '{SEVERITY_LABEL}':")
print(f"  Train speakers : {len(train_speakers):3d}  →  {len(train_ds):,} samples")
print(f"  Val   speakers : {len(val_speakers):3d}  →  {len(val_ds):,} samples")
print(f"  Speaker overlap: {len(_overlap)} ✓ (zero required)")
print(f"  Train > Val    : {len(train_ds):,} > {len(val_ds):,} ✓")
print(f"  Test           : {len(test_ds):,} samples (all severities, untouched)")

  train/train-00000-of-00008.parquet ... 

data/train-00000-of-00008.parquet:   0%|          | 0.00/443M [00:00<?, ?B/s]

OK
  train/train-00001-of-00008.parquet ... 

data/train-00001-of-00008.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

OK
  train/train-00002-of-00008.parquet ... 

...

Filter:   0%|          | 0/6230 [00:00<?, ? examples/s]

  Filtered 6,230 -> 2,971 samples for ['severe']


Filter:   0%|          | 0/2971 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2971 [00:00<?, ? examples/s]


Speaker-disjoint split for 'severe':
  Train speakers :  16  →  2,257 samples
  Val   speakers :   4  →  714 samples
  Speaker overlap: 0 ✓ (zero required)
  Train > Val    : 2,257 > 714 ✓
  Test           : 1,013 samples (all severities, untouched)


## Cell 7: Load Whisper processor

In [7]:
print(f"Loading Whisper processor: {WHISPER_MODEL_TYPE} ...")
feature_extractor = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL_TYPE)
tokenizer  = WhisperTokenizer.from_pretrained(WHISPER_MODEL_TYPE, language=LANGUAGE, task=TASK)
processor  = WhisperProcessor.from_pretrained(WHISPER_MODEL_TYPE, language=LANGUAGE, task=TASK)

train_ds = train_ds.cast_column(AUDIO_COL, Audio(sampling_rate=SAMPLE_RATE))
val_ds   = val_ds.cast_column(AUDIO_COL,   Audio(sampling_rate=SAMPLE_RATE))
test_ds  = test_ds.cast_column(AUDIO_COL,  Audio(sampling_rate=SAMPLE_RATE))
print(f"Processor loaded. Audio resampled to {SAMPLE_RATE} Hz on access.")

Loading Whisper processor: openai/whisper-large-v3 ...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Processor loaded. Audio resampled to 16000 Hz on access.


## Cell 8: Preprocess & augment

In [8]:
import random

DISFLUENCY_RE = re.compile(r"\b(uh+|um+|er+|hmm+|ah+)\b", re.IGNORECASE)

def normalise(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

class PrepareTrainSample:
    def __init__(self, augment_config, feature_extractor, tokenizer,
                 audio_col, transcript_col, aug_prob, use_specaugment):
        self.augment_config    = augment_config
        self.feature_extractor = feature_extractor
        self.tokenizer         = tokenizer
        self.audio_col         = audio_col
        self.transcript_col    = transcript_col
        self.aug_prob          = aug_prob
        self.use_specaugment   = use_specaugment
        self._augment          = None

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_augment"] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)

    def _build_augment(self):
        from audiomentations import (
            Compose, AddGaussianNoise, AddGaussianSNR,
            PitchShift, TimeStretch, Gain, LowPassFilter, RoomSimulator,
        )
        c = self.augment_config
        return Compose([
            AddGaussianSNR(min_snr_db=c["min_snr_db"], max_snr_db=c["max_snr_db"], p=c["p_snr"]),
            AddGaussianNoise(min_amplitude=c["min_amp"], max_amplitude=c["max_amp"], p=c["p_noise"]),
            PitchShift(min_semitones=c["min_semi"], max_semitones=c["max_semi"], p=c["p_pitch"]),
            TimeStretch(min_rate=c["min_rate"], max_rate=c["max_rate"], p=c["p_stretch"],
                        leave_length_unchanged=False),
            RoomSimulator(
                min_size_x=c["min_sx"],   max_size_x=c["max_sx"],
                min_size_y=c["min_sy"],   max_size_y=c["max_sy"],
                min_size_z=c["min_sz"],   max_size_z=c["max_sz"],
                min_source_x=c["min_srcx"], max_source_x=c["max_srcx"],
                min_source_y=c["min_srcy"], max_source_y=c["max_srcy"],
                min_source_z=c["min_srcz"], max_source_z=c["max_srcz"],
                min_mic_distance=c["min_mic"], max_mic_distance=c["max_mic"],
                p=c["p_room"],
            ),
            LowPassFilter(min_cutoff_freq=c["min_lp"], max_cutoff_freq=c["max_lp"], p=c["p_lp"]),
            Gain(min_gain_db=c["min_gain"], max_gain_db=c["max_gain"], p=c["p_gain"]),
        ])

    @property
    def augment(self):
        if self._augment is None:
            self._augment = self._build_augment()
        return self._augment

    @staticmethod
    def _spec_augment(features, freq_mask_num=2, freq_mask_width=10,
                      time_mask_num=2, time_mask_width=20):
        F, T = features.shape[0], features.shape[1]
        feat = features.copy()
        for _ in range(freq_mask_num):
            f  = np.random.randint(0, freq_mask_width)
            f0 = np.random.randint(0, max(1, F - f))
            feat[f0:f0+f, :] = 0.0
        for _ in range(time_mask_num):
            t  = np.random.randint(0, time_mask_width)
            t0 = np.random.randint(0, max(1, T - t))
            feat[:, t0:t0+t] = 0.0
        return feat

    def __call__(self, batch):
        audio = batch[self.audio_col]
        arr   = audio["array"].astype(np.float32)
        sr    = audio["sampling_rate"]
        if random.random() < self.aug_prob:
            try:
                arr = self.augment(samples=arr, sample_rate=sr)
            except Exception:
                pass
        feats = self.feature_extractor(
            arr, sampling_rate=sr, return_tensors="pt"
        ).input_features[0].numpy()
        if self.use_specaugment and random.random() < self.aug_prob:
            feats = self._spec_augment(feats)
        batch["input_features"] = feats
        batch["labels"] = self.tokenizer(normalise(str(batch[self.transcript_col]))).input_ids
        return batch

prepare_train_sample = PrepareTrainSample(
    augment_config=AUGMENT_CONFIG, feature_extractor=feature_extractor,
    tokenizer=tokenizer, audio_col=AUDIO_COL, transcript_col=TRANSCRIPT_COL,
    aug_prob=AUG_PROB, use_specaugment=True,
)

def prepare_eval_sample(batch):
    audio = batch[AUDIO_COL]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"],
        return_tensors="pt", return_attention_mask=True,
    ).input_features[0]
    batch["labels"] = tokenizer(normalise(str(batch[TRANSCRIPT_COL]))).input_ids
    return batch

keep_cols = ["input_features", "labels"]

print(f"Preprocessing {SEVERITY_LABEL} training set ({len(train_ds):,} samples)...")
train_prep = train_ds.map(
    prepare_train_sample,
    remove_columns=[c for c in train_ds.column_names if c not in keep_cols],
    num_proc=1, desc=f"train-{SEVERITY_LABEL}",
)

print("Preprocessing validation set...")
val_prep = val_ds.map(
    prepare_eval_sample,
    remove_columns=[c for c in val_ds.column_names if c not in keep_cols],
    num_proc=1, desc="validation",
)

print(f"  Train : {len(train_prep):,}  |  Val : {len(val_prep):,}")

Preprocessing severe training set (2,257 samples)...


train-severe:   0%|          | 0/2257 [00:00<?, ? examples/s]

Preprocessing validation set...


validation:   0%|          | 0/714 [00:00<?, ? examples/s]

  Train : 2,257  |  Val : 714


## Cell 9: Data collator & metrics

In [9]:
@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features],
            return_tensors="pt",
        )
        if "attention_mask" not in batch:
            batch["attention_mask"] = torch.ones(
                batch["input_features"].shape[0],
                batch["input_features"].shape[2],
                dtype=torch.long,
                device=batch["input_features"].device,
            )
        labels_batch = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features],
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels

        # Explicitly build decoder_input_ids so gradient-checkpointed LoRA
        # forward passes receive them (newer PEFT drops labels before shift_tokens_right).
        from transformers.models.whisper.modeling_whisper import shift_tokens_right as _str
        _pad = self.processor.tokenizer.pad_token_id
        _bos = self.processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
        _lbl = labels.clone()
        _lbl[_lbl == -100] = _pad
        batch["decoder_input_ids"] = _str(_lbl, _pad, _bos)

        return batch

data_collator = DataCollatorSpeechSeq2Seq(processor=processor)
wer_metric    = evaluate.load("wer")
cer_metric    = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = [normalise(s) for s in tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)]
    label_str = [normalise(s) for s in tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    wer = round(100 * wer_metric.compute(predictions=pred_str, references=label_str), 2)
    cer = round(100 * cer_metric.compute(predictions=pred_str, references=label_str), 2)
    return {"wer": wer, "cer": cer}

print("Data collator and metrics ready.")

Data collator and metrics ready.


## Cell 10: Build LoRA model

In [10]:
print(f"Loading base model: {WHISPER_MODEL_TYPE}  (rank={LORA_R}, alpha={LORA_ALPHA})...")
sys.stdout.flush()

base = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_TYPE)
base.config.forced_decoder_ids            = None
base.config.suppress_tokens               = []
base.generation_config.forced_decoder_ids = None
base.generation_config.task               = TASK
base.generation_config.language           = LANGUAGE
base.config.pad_token_id                  = base.config.eos_token_id
base.config.use_cache                     = False

all_module_names = {name.split(".")[-1] for name, _ in base.named_modules()}
valid_targets = [t for t in LORA_TARGETS if t in all_module_names]
print(f"LoRA targets: {valid_targets}")

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=valid_targets, bias="none",
)
model = get_peft_model(base, lora_config)
model.enable_input_require_grads()

model.generation_config.forced_decoder_ids   = None
model.generation_config.task                 = TASK
model.generation_config.language             = LANGUAGE
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.repetition_penalty   = 1.3
model.generation_config.num_beams            = 5
model.generation_config.max_length           = MAX_GEN_LEN
model.config.pad_token_id                    = model.config.eos_token_id

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,}  ({100*trainable/total:.2f}%)")
_ = model.to(device)

Loading base model: openai/whisper-large-v3  (rank=32, alpha=64)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

LoRA targets: ['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2']
Trainable: 57,671,680  (3.60%)


## Cell 11: Train — Severe impairment

In [ ]:
print("=" * 60)
print("TRAINING  —  severe impairment only")
print("=" * 60)

args = Seq2SeqTrainingArguments(
    output_dir                    = OUTPUT_DIR,
    num_train_epochs              = MAX_EPOCHS,
    per_device_train_batch_size   = BATCH_SIZE,
    per_device_eval_batch_size    = EVAL_BATCH_SIZE,
    gradient_accumulation_steps   = GRAD_ACCUM,
    learning_rate                 = LEARNING_RATE,
    warmup_steps                  = LR_WARMUP_STEPS,
    lr_scheduler_type             = "cosine",
    bf16                          = USE_BF16,
    fp16                          = USE_FP16,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    eval_strategy                 = "steps",
    eval_steps                    = 200,
    save_strategy                 = "steps",
    save_steps                    = 200,
    save_total_limit              = NUM_CHECKPOINTS,
    load_best_model_at_end        = True,
    metric_for_best_model         = "wer",
    greater_is_better             = False,
    predict_with_generate         = True,
    generation_max_length         = MAX_GEN_LEN,
    generation_num_beams          = 5,
    weight_decay                  = 0.01,
    label_smoothing_factor        = 0.1,
    logging_steps                 = LOGGING_STEPS,
    report_to                     = ["tensorboard"],
    dataloader_num_workers        = 0,
    remove_unused_columns         = False,
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = args,
    train_dataset    = train_prep,
    eval_dataset     = val_prep,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = processor.feature_extractor,
    callbacks        = [DeferredEarlyStopping(
        min_epochs               = 4,
        early_stopping_patience  = EARLY_STOP_PAT,
        early_stopping_threshold = 0.001,
    )],
)

print(f"Training on {len(train_prep):,} severe samples")
result = trainer.train()
eval_result = trainer.evaluate()
print(f"\nTraining complete")
print(f"  Train loss : {result.training_loss:.4f}")
print(f"  Val WER    : {eval_result['eval_wer']:.2f}%")
print(f"  Val CER    : {eval_result['eval_cer']:.2f}%")

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


TRAINING  —  severe impairment only
Training on 2,257 severe samples


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss,Wer,Cer
200,2.238100,2.177875,31.660000,22.350000
400,2.059600,2.077962,30.410000,21.330000


## Cell 12: Evaluate on test split

In [ ]:
model.eval()
refs_all, hyps_all, speakers_list = [], [], []
by_prompt  = defaultdict(lambda: {"refs": [], "hyps": []})
by_severity = defaultdict(lambda: {"refs": [], "hyps": []})

from better_profanity import profanity as _profanity_filter
_profanity_filter.load_censor_words()

def clean_transcript(text: str) -> str:
    """Censor profane words using better-profanity (community-maintained word list)."""
    censored = _profanity_filter.censor(text)
    # Replace runs of asterisks with a readable placeholder
    return re.sub(r"\*+", "[inaudible]", censored)

print(f"Running inference on {len(test_ds):,} test samples...")

for row in tqdm(test_ds, desc="Evaluation"):
    audio   = row[AUDIO_COL]
    ref     = normalise(str(row[TRANSCRIPT_COL]))
    speaker = str(row.get(SPEAKER_COL, "unknown"))
    ptype   = str(row.get(PROMPT_TYPE_COL, "unknown"))
    sev     = spk_to_severity.get(speaker, "unknown")

    processed = processor(
        audio["array"], sampling_rate=audio["sampling_rate"],
        return_tensors="pt", return_attention_mask=True,
    )
    inputs    = processed.input_features.to(device)
    attn_mask = processed.get("attention_mask",
                    torch.ones(inputs.shape[:2], dtype=torch.long)).to(device)

    with torch.no_grad():
        ids = model.generate(
            inputs, attention_mask=attn_mask,
            language=LANGUAGE, task=TASK,
            num_beams=5, no_repeat_ngram_size=3,
        )
    hyp = normalise(clean_transcript(processor.tokenizer.decode(ids[0], skip_special_tokens=True)))

    refs_all.append(ref); hyps_all.append(hyp); speakers_list.append(speaker)
    by_prompt[ptype]["refs"].append(ref);    by_prompt[ptype]["hyps"].append(hyp)
    by_severity[sev]["refs"].append(ref);    by_severity[sev]["hyps"].append(hyp)

overall_wer = 100 * wer_metric.compute(predictions=hyps_all, references=refs_all)
overall_cer = 100 * cer_metric.compute(predictions=hyps_all, references=refs_all)
print(f"\n{'='*55}")
print(f"OVERALL  WER: {overall_wer:.1f}%   CER: {overall_cer:.1f}%  (n={len(test_ds):,})")
print(f"{'='*55}")

print("\nWER by severity:")
for sev, data in sorted(by_severity.items()):
    w = 100 * wer_metric.compute(predictions=data["hyps"], references=data["refs"])
    c = 100 * cer_metric.compute(predictions=data["hyps"], references=data["refs"])
    print(f"  {sev:55s}: WER={w:.1f}%  CER={c:.1f}%  (n={len(data['refs'])})")

print("\nWER by prompt type:")
for ptype, data in sorted(by_prompt.items()):
    w = 100 * wer_metric.compute(predictions=data["hyps"], references=data["refs"])
    print(f"  {ptype:20s}: {w:.1f}%  (n={len(data['refs'])})")

## Cell 13: Save merged model & LoRA adapter

In [ ]:
import json as _json

adapter_dir = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(adapter_dir, exist_ok=True)
print(f"Saving LoRA adapter -> {adapter_dir}")
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)

print("Merging LoRA weights into base model...")
merged = model.merge_and_unload()
merged.save_pretrained(OUTPUT_DIR, safe_serialization=True)
processor.save_pretrained(OUTPUT_DIR)

summary = {
    "base_model"      : WHISPER_MODEL_TYPE,
    "severity_target" : SEVERITY_LABEL,
    "dataset"         : DATASET_NAME,
    "lora_rank"       : LORA_R,
    "lora_alpha"      : LORA_ALPHA,
    "lora_targets"    : valid_targets,
    "aug_prob"        : AUG_PROB,
    "overall_test_wer": round(overall_wer, 2),
    "overall_test_cer": round(overall_cer, 2),
    "train_samples"   : len(train_prep),
    "val_samples"     : len(val_prep),
    "test_samples"    : len(test_ds),
}
with open(os.path.join(OUTPUT_DIR, "eval_summary.json"), "w") as f:
    _json.dump(summary, f, indent=2)
print(f"\nSaved to {OUTPUT_DIR}")
print(f"  Overall test WER : {overall_wer:.1f}%  CER : {overall_cer:.1f}%")

## Cell 14: Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=REPO_NAME, exist_ok=True, private=True, repo_type="model")
api.create_repo(repo_id=REPO_NAME + "-lora-adapter", exist_ok=True, private=True, repo_type="model")

merged.push_to_hub(REPO_NAME, private=True, token=HF_TOKEN)
processor.push_to_hub(REPO_NAME, private=True, token=HF_TOKEN)

api.upload_folder(
    folder_path=adapter_dir,
    repo_id=REPO_NAME + "-lora-adapter",
    commit_message=f"Upload LoRA adapter — {SEVERITY_LABEL} model",
    token=HF_TOKEN,
)
print(f"Pushed merged model to  : {REPO_NAME}")
print(f"Pushed LoRA adapter to  : {REPO_NAME}-lora-adapter")